In [1]:
import sys
import yaml
import pandas as pd
import os

sys.path.append("../..")

In [2]:
    # Load config
config_path = os.path.join("..", "..", "config", "analysis_config.yaml")
INTERIM_DATA_FOLDER = os.path.join("..", "..", "data", "interim")

with open(config_path, "r") as f:
    analysis_cfg = yaml.safe_load(f)

ANALYSIS_START = pd.to_datetime(analysis_cfg.get("start_date", "2000-01-01"))
ANALYSIS_END = pd.to_datetime(analysis_cfg.get("end_date", "2025-12-31"))

# Optional tickers override
ANALYSIS_TICKERS = analysis_cfg.get(
        "tickers", None
    )  # None means use all downloaded tickers

# Filter feature data by date range
filtered_data = {}
features_files = [
        f for f in os.listdir(INTERIM_DATA_FOLDER) if f.endswith("_features.csv")
    ]

for file in features_files:
        ticker = os.path.basename(file).split("_")[0]
        if ANALYSIS_TICKERS and ticker not in ANALYSIS_TICKERS:
            continue
        df = pd.read_csv(
            os.path.join(INTERIM_DATA_FOLDER, file), index_col=0, parse_dates=True
        )
        df_filtered = df.loc[(df.index >= ANALYSIS_START) & (df.index <= ANALYSIS_END)]
        filtered_data[ticker] = df_filtered
        # Optional: save filtered version
        for ticker, df_filtered in filtered_data.items():
            df_filtered.to_csv(
                os.path.join(INTERIM_DATA_FOLDER, f"{ticker}_features_filtered.csv")
            )

In [6]:
from src.utils.efficient_frontier_utils import compute_and_save_efficient_frontier


ef_cfg = analysis_cfg.get("efficient_frontier", {})
ef_weights = None
returns_df = None
if ef_cfg.get("enabled", False):
        print("\nStep 4b: Computing Efficient Frontier...")
        ef_weights, returns_df = compute_and_save_efficient_frontier(
            filtered_data, ef_cfg
        )
print(ef_weights)




Step 4b: Computing Efficient Frontier...
Efficient Frontier saved to C:/Users/eduar/Projects/Python/quant_project/reports/figures/efficient_frontier.png
{'AAPL': np.float64(0.0), 'ABT': np.float64(0.8528283117920804), 'BAC': np.float64(0.0), 'C': np.float64(7.529505872577783e-18), 'DIS': np.float64(8.325663033780263e-19), 'GOOGL': np.float64(2.1000445849995447e-18), 'INTC': np.float64(0.11277426598666163), 'MSFT': np.float64(0.003310096204680252), 'WFC': np.float64(0.031087326016577495), 'WMT': np.float64(1.0408340855860843e-17)}


In [10]:
import json
import os

# Build path relative to main.py
cwd = os.getcwd()
reports_dir = os.path.join(cwd,"..","..","reports")
os.makedirs(reports_dir, exist_ok=True)

# File path
json_path = os.path.join(reports_dir, "ef_weights.json")

# Save ef_weights dict
with open(json_path, "w") as f:
    json.dump({k: float(v) for k, v in ef_weights.items()}, f, indent=4)

print(f"Efficient frontier weights saved to {json_path}")

Efficient frontier weights saved to c:\Users\eduar\Projects\Python\quant_project\notebooks\exploration\..\..\reports\ef_weights.json


In [4]:
from src.portfolio.tracker import MultiClientPortfolioTracker


print("\nStep 8: Managing client portfolios...")
tracker = MultiClientPortfolioTracker()

# Add clients
tracker.add_client(
        "Eduardo",
        initial_capital=7.41,
        initial_positions={
            "AAPL": 0.004460,
            "WMT": 0.010396,
            "DIS": 0.008504,
            "ABT": 0.007578,
            "GOOGL": 0.010531,
        },
    )
tracker.add_client(
        "Client1", initial_capital=50000, initial_positions={"INTC": 100, "AAPL": 20}
    )

# Get current prices from latest filtered data
current_prices = {
        ticker: df["Close"].iloc[-1] for ticker, df in filtered_data.items()
    }
tracker.update_all_market_prices(current_prices)

# Rebalance to Efficient Frontier weights
if ef_weights:
        for client_name in tracker.get_all_clients():
            trades = tracker.rebalance_client(client_name, ef_weights, current_prices)
            print(f"Trades for {client_name}:", trades)
            # Execute trades
            for ticker, shares in trades.items():
                if shares > 0:
                    tracker.get_client(client_name).input_trade(
                        ticker, shares, current_prices[ticker], "buy"
                    )
                elif shares < 0:
                    tracker.get_client(client_name).input_trade(
                        ticker, -shares, current_prices[ticker], "sell"
                    )

# Save all histories after rebalancing
tracker.save_all_histories()
print("Client portfolio histories saved.")


Step 8: Managing client portfolios...
Trades for Eduardo: {'AAPL': np.float64(-0.00446), 'ABT': np.float64(0.02617308249415658), 'BAC': np.float64(0.0), 'C': np.float64(6.265193354264361e-19), 'DIS': np.float64(-0.008504), 'GOOGL': np.float64(-0.010531), 'INTC': np.float64(0.025709181840263995), 'MSFT': np.float64(4.5607187282688366e-05), 'WFC': np.float64(0.0024731502389577086), 'WMT': np.float64(-0.010396)}
Trades for Client1: {'AAPL': np.float64(-20.0), 'ABT': np.float64(41.53694631313761), 'BAC': np.float64(0.0), 'C': np.float64(7.710478620724562e-16), 'DIS': np.float64(5.962676500432432e-17), 'GOOGL': np.float64(8.792446085769367e-17), 'INTC': np.float64(-68.36009908604788), 'MSFT': np.float64(0.05612807500269715), 'WFC': np.float64(3.0436685613771126), 'WMT': np.float64(7.557305684395851e-16)}
Client portfolio histories saved.


c:\Users\eduar\Projects\Python\quant_project\notebooks\exploration\../..\src\portfolio\tracker.py:44: FutureWarning: The behavior of DataFrame concatenation with empty or all-NA entries is deprecated. In a future version, this will no longer exclude empty or all-NA columns when determining the result dtypes. To retain the old behavior, exclude the relevant entries before the concat operation.
  self.history = pd.concat(
